##True Coding

In [1]:
!pip -q install tensorflow scikit-learn pandas numpy

In [2]:
import os
import json
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import tensorflow as tf

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import classification_report, f1_score, accuracy_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

TensorFlow: 2.19.0


In [3]:
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)

print("Shape:", df.shape)
display(df.head())
print(df.columns.tolist())

Saving adult.csv to adult.csv
Shape: (8000, 14)


,age_years,sex,heart_rate_bpm,respiratory_rate_bpm,systolic_bp_mmHg,spo2_percent,temperature_c,level_of_consciousness,chief_complaint_category,duration_days,comorbidity_count,pain_distress_score_0_10,clinical_disposition,severity_score
0,57,Female,120.4,23.1,117.6,94.0,36.8,Alert,Respiratory,0,1,1,Treat + monitor,Medium
1,70,Female,127.7,23.2,120.5,96.1,38.7,Alert,Fever/Infection,2,1,9,Treat + monitor,Medium
2,55,Female,90.0,17.2,131.9,97.3,37.6,Alert,Urinary,3,1,0,Treat locally,Low
3,64,Female,84.4,20.9,107.8,93.8,38.3,Alert,Chest pain,3,1,10,Treat + monitor,Medium
4,52,Male,100.3,30.7,99.2,89.7,37.5,Alert,Respiratory,3,1,6,Stabilize + refer,High


['age_years', 'sex', 'heart_rate_bpm', 'respiratory_rate_bpm', 'systolic_bp_mmHg', 'spo2_percent', 'temperature_c', 'level_of_consciousness', 'chief_complaint_category', 'duration_days', 'comorbidity_count', 'pain_distress_score_0_10', 'clinical_disposition', 'severity_score']


In [4]:
expected_cols = [
    'age_years',
    'sex',
    'heart_rate_bpm',
    'respiratory_rate_bpm',
    'systolic_bp_mmHg',
    'spo2_percent',
    'temperature_c',
    'level_of_consciousness',
    'chief_complaint_category',
    'duration_days',
    'comorbidity_count',
    'pain_distress_score_0_10',
    'clinical_disposition',
    'severity_score'
]

missing_cols = [c for c in expected_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

df = df[expected_cols].copy().drop_duplicates().reset_index(drop=True)

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

print("Cleaned shape:", df.shape)
display(df.head())

Cleaned shape: (8000, 14)


,age_years,sex,heart_rate_bpm,respiratory_rate_bpm,systolic_bp_mmHg,spo2_percent,temperature_c,level_of_consciousness,chief_complaint_category,duration_days,comorbidity_count,pain_distress_score_0_10,clinical_disposition,severity_score
0,57,Female,120.4,23.1,117.6,94.0,36.8,Alert,Respiratory,0,1,1,Treat + monitor,Medium
1,70,Female,127.7,23.2,120.5,96.1,38.7,Alert,Fever/Infection,2,1,9,Treat + monitor,Medium
2,55,Female,90.0,17.2,131.9,97.3,37.6,Alert,Urinary,3,1,0,Treat locally,Low
3,64,Female,84.4,20.9,107.8,93.8,38.3,Alert,Chest pain,3,1,10,Treat + monitor,Medium
4,52,Male,100.3,30.7,99.2,89.7,37.5,Alert,Respiratory,3,1,6,Stabilize + refer,High


In [5]:
clinical_order = [
    'Treat locally',
    'Treat + monitor',
    'Stabilize + refer',
    'Emergency referral'
]

severity_order = [
    'Low',
    'Medium',
    'High'
]

clinical_map = {label: idx for idx, label in enumerate(clinical_order)}
severity_map = {label: idx for idx, label in enumerate(severity_order)}

inv_clinical = {v: k for k, v in clinical_map.items()}
inv_severity = {v: k for k, v in severity_map.items()}

bad_clinical = sorted(set(df['clinical_disposition']) - set(clinical_order))
bad_severity = sorted(set(df['severity_score']) - set(severity_order))

if bad_clinical:
    raise ValueError(f"Unexpected clinical labels: {bad_clinical}")
if bad_severity:
    raise ValueError(f"Unexpected severity labels: {bad_severity}")

df["clinical_disposition_encoded"] = df["clinical_disposition"].map(clinical_map)
df["severity_score_encoded"] = df["severity_score"].map(severity_map)

print(df[["clinical_disposition", "clinical_disposition_encoded",
          "severity_score", "severity_score_encoded"]].head())

  clinical_disposition  clinical_disposition_encoded severity_score  \
0      Treat + monitor                             1         Medium   
1      Treat + monitor                             1         Medium   
2        Treat locally                             0            Low   
3      Treat + monitor                             1         Medium   
4    Stabilize + refer                             2           High   

   severity_score_encoded  
0                       1  
1                       1  
2                       0  
3                       1  
4                       2  


In [6]:
feature_cols = [
    'age_years',
    'sex',
    'heart_rate_bpm',
    'respiratory_rate_bpm',
    'systolic_bp_mmHg',
    'spo2_percent',
    'temperature_c',
    'level_of_consciousness',
    'chief_complaint_category',
    'duration_days',
    'comorbidity_count',
    'pain_distress_score_0_10'
]

numeric_cols = [
    'age_years',
    'heart_rate_bpm',
    'respiratory_rate_bpm',
    'systolic_bp_mmHg',
    'spo2_percent',
    'temperature_c',
    'duration_days',
    'comorbidity_count',
    'pain_distress_score_0_10'
]

categorical_cols = [
    'sex',
    'level_of_consciousness',
    'chief_complaint_category'
]

X = df[feature_cols].copy()
y_disp = df["clinical_disposition_encoded"].values
y_sev = df["severity_score_encoded"].values

stratify_key = (
    df["clinical_disposition"].astype(str) + " | " + df["severity_score"].astype(str)
)

In [7]:
X_train_df, X_test_df, y_disp_train, y_disp_test, y_sev_train, y_sev_test = train_test_split(
    X,
    y_disp,
    y_sev,
    test_size=0.20,
    random_state=SEED,
    stratify=stratify_key
)

print("Train:", X_train_df.shape)
print("Test :", X_test_df.shape)

Train: (6400, 12)
Test : (1600, 12)


In [8]:
# Latest sklearn uses sparse_output=False
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
scaler = StandardScaler()

X_train_num = scaler.fit_transform(X_train_df[numeric_cols])
X_test_num = scaler.transform(X_test_df[numeric_cols])

X_train_cat = ohe.fit_transform(X_train_df[categorical_cols])
X_test_cat = ohe.transform(X_test_df[categorical_cols])

X_train = np.hstack([X_train_num, X_train_cat]).astype("float32")
X_test = np.hstack([X_test_num, X_test_cat]).astype("float32")

input_dim = X_train.shape[1]

print("Numeric shape:", X_train_num.shape)
print("Categorical shape:", X_train_cat.shape)
print("Final train shape:", X_train.shape)
print("Final test shape :", X_test.shape)

Numeric shape: (6400, 9)
Categorical shape: (6400, 16)
Final train shape: (6400, 25)
Final test shape : (1600, 25)


In [9]:
def build_model_1(input_dim, n_disp=4, n_sev=3):
    inputs = tf.keras.Input(shape=(input_dim,), name="adult_features")

    x = tf.keras.layers.Dense(64, activation="relu")(inputs)
    x = tf.keras.layers.Dense(32, activation="relu")(x)

    disp_out = tf.keras.layers.Dense(n_disp, activation="softmax", name="clinical_disposition")(x)
    sev_out = tf.keras.layers.Dense(n_sev, activation="softmax", name="severity_score")(x)

    model = tf.keras.Model(inputs=inputs, outputs=[disp_out, sev_out], name="adult_model_v1")
    return model


def build_model_2(input_dim, n_disp=4, n_sev=3):
    inputs = tf.keras.Input(shape=(input_dim,), name="adult_features")

    x = tf.keras.layers.Dense(128, activation="relu")(inputs)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dense(32, activation="relu")(x)

    disp_out = tf.keras.layers.Dense(n_disp, activation="softmax", name="clinical_disposition")(x)
    sev_out = tf.keras.layers.Dense(n_sev, activation="softmax", name="severity_score")(x)

    model = tf.keras.Model(inputs=inputs, outputs=[disp_out, sev_out], name="adult_model_v2")
    return model


def build_model_3(input_dim, n_disp=4, n_sev=3):
    inputs = tf.keras.Input(shape=(input_dim,), name="adult_features")

    x = tf.keras.layers.Dense(128, activation="relu")(inputs)
    x = tf.keras.layers.Dropout(0.30)(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.20)(x)
    x = tf.keras.layers.Dense(32, activation="relu")(x)

    disp_out = tf.keras.layers.Dense(n_disp, activation="softmax", name="clinical_disposition")(x)
    sev_out = tf.keras.layers.Dense(n_sev, activation="softmax", name="severity_score")(x)

    model = tf.keras.Model(inputs=inputs, outputs=[disp_out, sev_out], name="adult_model_v3")
    return model

In [10]:
def compile_model(model):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss={
            "clinical_disposition": "sparse_categorical_crossentropy",
            "severity_score": "sparse_categorical_crossentropy"
        },
        metrics={
            "clinical_disposition": ["accuracy"],
            "severity_score": ["accuracy"]
        }
    )
    return model

models_dict = {
    "model_1_simple_dense": compile_model(build_model_1(input_dim)),
    "model_2_deeper_dense": compile_model(build_model_2(input_dim)),
    "model_3_dropout_dense": compile_model(build_model_3(input_dim))
}

for name, model in models_dict.items():
    print("\n", "="*80)
    print(name)
    model.summary()


model_1_simple_dense


Model: "adult_model_v1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ adult_features      │ (None, 25)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      1,664 │ adult_features[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │      2,080 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ clinical_dispositi… │ (None, 4)         │        132 │ dense_1[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ severity_score      │ (None, 3)         │         99 │ dense_1[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,975 (15.53 KB)

 Trainable params: 3,975 (15.53 KB)

...

Model: "adult_model_v3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ adult_features      │ (None, 25)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 128)       │      3,328 │ adult_features[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 64)        │      8,256 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 32)        │      2,080 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ clinical_dispositi… │ (None, 4)         │        132 │ dense_7[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ severity_score      │ (None, 3)         │         99 │ dense_7[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 13,895 (54.28 KB)

 Trainable params: 13,895 (54.28 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=6,
    restore_best_weights=True
)

histories = {}
trained_models = {}

for name, model in models_dict.items():
    print("\nTraining:", name)

    history = model.fit(
        X_train,
        {
            "clinical_disposition": y_disp_train,
            "severity_score": y_sev_train
        },
        validation_split=0.15,
        epochs=40,
        batch_size=32,
        callbacks=[early_stop],
        verbose=1
    )

    histories[name] = history.history
    trained_models[name] = model


Training: model_1_simple_dense
Epoch 1/40
170/170 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - clinical_disposition_accuracy: 0.5811 - clinical_disposition_loss: 0.9454 - loss: 1.5792 - severity_score_accuracy: 0.7252 - severity_score_loss: 0.6339 - val_clinical_disposition_accuracy: 0.7281 - val_clinical_disposition_loss: 0.6388 - val_loss: 0.9909 - val_severity_score_accuracy: 0.8604 - val_severity_score_loss: 0.3520
Epoch 2/40
170/170 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - clinical_disposition_accuracy: 0.7629 - clinical_disposition_loss: 0.5754 - loss: 0.9210 - severity_score_accuracy: 0.8561 - severity_score_loss: 0.3455 - val_clinical_disposition_accuracy: 0.7969 - val_clinical_disposition_loss: 0.5100 - val_loss: 0.8030 - val_severity_score_accuracy: 0.8781 - val_severity_score_loss: 0.2930
Epoch 3/40
170/170 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - clinical_disposition_accuracy: 0.7987 - clinical_disposition_loss: 0.4878 - loss: 0.7954 - severity_score_accuracy: 0.8706 - severity_score_loss: 0.30

In [12]:
def evaluate_model(model, X_test, y_disp_test, y_sev_test, name):
    pred_disp_probs, pred_sev_probs = model.predict(X_test, verbose=0)

    pred_disp = np.argmax(pred_disp_probs, axis=1)
    pred_sev = np.argmax(pred_sev_probs, axis=1)

    disp_acc = accuracy_score(y_disp_test, pred_disp)
    sev_acc = accuracy_score(y_sev_test, pred_sev)

    disp_f1 = f1_score(y_disp_test, pred_disp, average="macro")
    sev_f1 = f1_score(y_sev_test, pred_sev, average="macro")

    composite = 0.65 * disp_f1 + 0.35 * sev_f1

    print("\n" + "="*90)
    print("MODEL:", name)
    print("="*90)
    print(f"Disposition Accuracy : {disp_acc:.4f}")
    print(f"Severity Accuracy    : {sev_acc:.4f}")
    print(f"Disposition Macro F1 : {disp_f1:.4f}")
    print(f"Severity Macro F1    : {sev_f1:.4f}")
    print(f"Composite Score      : {composite:.4f}")

    print("\nDisposition Report")
    print(classification_report(
        y_disp_test,
        pred_disp,
        target_names=clinical_order,
        digits=4
    ))

    print("\nSeverity Report")
    print(classification_report(
        y_sev_test,
        pred_sev,
        target_names=severity_order,
        digits=4
    ))

    return {
        "model": name,
        "disposition_accuracy": disp_acc,
        "severity_accuracy": sev_acc,
        "disposition_f1_macro": disp_f1,
        "severity_f1_macro": sev_f1,
        "composite_score": composite
    }

results = []
for name, model in trained_models.items():
    result = evaluate_model(model, X_test, y_disp_test, y_sev_test, name)
    results.append(result)

results_df = pd.DataFrame(results).sort_values(
    by=["composite_score", "disposition_f1_macro", "severity_f1_macro"],
    ascending=False
).reset_index(drop=True)

print("\nFinal comparison:")
display(results_df)


MODEL: model_1_simple_dense
Disposition Accuracy : 0.8850
Severity Accuracy    : 0.9087
Disposition Macro F1 : 0.8851
Severity Macro F1    : 0.9069
Composite Score      : 0.8927

Disposition Report
                    precision    recall  f1-score   support

     Treat locally     0.9455    0.9100    0.9274       400
   Treat + monitor     0.8337    0.8650    0.8491       400
 Stabilize + refer     0.8414    0.8225    0.8319       400
Emergency referral     0.9218    0.9425    0.9320       400

          accuracy                         0.8850      1600
         macro avg     0.8856    0.8850    0.8851      1600
      weighted avg     0.8856    0.8850    0.8851      1600


Severity Report
              precision    recall  f1-score   support

         Low     0.9507    0.9401    0.9454       451
      Medium     0.8599    0.8351    0.8473       485
        High     0.9151    0.9413    0.9280       664

    accuracy                         0.9087      1600
   macro avg     0.9085    0.

,model,disposition_accuracy,severity_accuracy,disposition_f1_macro,severity_f1_macro,composite_score
0,model_1_simple_dense,0.885000,0.908750,0.885085,0.906881,0.892714
1,model_2_deeper_dense,0.764375,0.851250,0.764011,0.845729,0.792612
2,model_3_dropout_dense,0.735625,0.844375,0.728480,0.836534,0.766299


In [13]:
best_name = results_df.iloc[0]["model"]
best_model = trained_models[best_name]

print("Best model selected:", best_name)

Best model selected: model_1_simple_dense


In [14]:
# Save Keras model
best_model.save("adult_tf_best_model.keras")

# Save preprocessing metadata for later inference / Android-side replication
preprocessing_assets = {
    "feature_columns": feature_cols,
    "numeric_columns": numeric_cols,
    "categorical_columns": categorical_cols,
    "clinical_order": clinical_order,
    "severity_order": severity_order,
    "clinical_map": clinical_map,
    "severity_map": severity_map,
    "scaler_mean": scaler.mean_.tolist(),
    "scaler_scale": scaler.scale_.tolist(),
    "ohe_categories": {col: cats.tolist() for col, cats in zip(categorical_cols, ohe.categories_)}
}

with open("adult_preprocessing_assets.json", "w") as f:
    json.dump(preprocessing_assets, f, indent=2)

print("Saved:")
print("- adult_tf_best_model.keras")
print("- adult_preprocessing_assets.json")

Saved:
- adult_tf_best_model.keras
- adult_preprocessing_assets.json


In [15]:
# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
tflite_model = converter.convert()

with open("adult_tf_best_model.tflite", "wb") as f:
    f.write(tflite_model)

print("Saved: adult_tf_best_model.tflite")

Saved artifact at '/tmp/tmppedh7igl'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 25), dtype=tf.float32, name='adult_features')
Output Type:
  List[TensorSpec(shape=(None, 4), dtype=tf.float32, name=None), TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)]
Captures:
  134139894399824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134139894400784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134139894400208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134139894400592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134139894403664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134139894402896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134139894403088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134139894402128: TensorSpec(shape=(), dtype=tf.resource, name=None)
Saved: adult_tf_best_model.tflite


In [16]:
files.download("adult_tf_best_model.keras")
files.download("adult_tf_best_model.tflite")
files.download("adult_preprocessing_assets.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
def preprocess_single_adult_input(sample_dict, scaler, ohe, numeric_cols, categorical_cols):
    sample_df = pd.DataFrame([sample_dict])

    X_num = scaler.transform(sample_df[numeric_cols])
    X_cat = ohe.transform(sample_df[categorical_cols])

    X_final = np.hstack([X_num, X_cat]).astype("float32")
    return X_final

sample_patient = {
    'age_years': 60,
    'sex': 'Male',
    'heart_rate_bpm': 120,
    'respiratory_rate_bpm': 28,
    'systolic_bp_mmHg': 95,
    'spo2_percent': 90,
    'temperature_c': 38.5,
    'level_of_consciousness': 'Alert',
    'chief_complaint_category': 'Respiratory',
    'duration_days': 2,
    'comorbidity_count': 2,
    'pain_distress_score_0_10': 7
}

X_sample = preprocess_single_adult_input(
    sample_patient,
    scaler=scaler,
    ohe=ohe,
    numeric_cols=numeric_cols,
    categorical_cols=categorical_cols
)

pred_disp_probs, pred_sev_probs = best_model.predict(X_sample, verbose=0)

pred_disp_idx = int(np.argmax(pred_disp_probs, axis=1)[0])
pred_sev_idx = int(np.argmax(pred_sev_probs, axis=1)[0])

print("Predicted clinical_disposition:", inv_clinical[pred_disp_idx])
print("Predicted severity_score      :", inv_severity[pred_sev_idx])

print("\nDisposition probabilities:")
for i, p in enumerate(pred_disp_probs[0]):
    print(f"{clinical_order[i]}: {p:.4f}")

print("\nSeverity probabilities:")
for i, p in enumerate(pred_sev_probs[0]):
    print(f"{severity_order[i]}: {p:.4f}")

Predicted clinical_disposition: Stabilize + refer
Predicted severity_score      : High

Disposition probabilities:
Treat locally: 0.0000
Treat + monitor: 0.0001
Stabilize + refer: 0.9869
Emergency referral: 0.0130

Severity probabilities:
Low: 0.0000
Medium: 0.0012
High: 0.9988


In [18]:
# Optional: test TFLite inference in Colab before Android
interpreter = tf.lite.Interpreter(model_path="adult_tf_best_model.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input details:", input_details)
print("Output details:", output_details)

interpreter.set_tensor(input_details[0]['index'], X_sample.astype(np.float32))
interpreter.invoke()

out1 = interpreter.get_tensor(output_details[0]['index'])
out2 = interpreter.get_tensor(output_details[1]['index'])

# Output order can vary, so sort by last dimension
outputs = [out1, out2]
outputs_sorted = sorted(outputs, key=lambda x: x.shape[-1])

pred_sev_tflite = outputs_sorted[0]
pred_disp_tflite = outputs_sorted[1]

print("TFLite clinical_disposition:", clinical_order[int(np.argmax(pred_disp_tflite, axis=1)[0])])
print("TFLite severity_score      :", severity_order[int(np.argmax(pred_sev_tflite, axis=1)[0])])

Input details: [{'name': 'serving_default_adult_features:0', 'index': 0, 'shape': array([ 1, 25], dtype=int32), 'shape_signature': array([-1, 25], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
Output details: [{'name': 'StatefulPartitionedCall_1:1', 'index': 14, 'shape': array([1, 3], dtype=int32), 'shape_signature': array([-1,  3], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}, {'name': 'StatefulPartitionedCall_1:0', 'index': 12, 'shape': array([1, 4], dtype=int32), 'shape_signature': array([-1,  4], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], 